In [8]:
import numpy as np
import pandas as pd

# 1. Đọc dữ liệu từ file Excel
file_path = r"E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\Sample Data.xlsx"
df = pd.read_excel(file_path)

# ==========================================
# 2. XỬ LÝ DỮ LIỆU KHUYẾT THIẾU (NULL VALUES)
# ==========================================
# 44 dòng null ở 'Resource Group' đều thuộc về nhân viên 'Emp0000403'.
# Tương tự, 65 dòng null ở 'Country' thuộc về 'Emp0000403' và 'Emp0000233'.
# Mỗi nhân viên có thông tin cố định ở các bản ghi khác, do đó ta tạo bảng tra cứu (mapping) theo mã Resource:

# Tạo dictionary ánh xạ từ các dòng không null
rg_map = (
    df.dropna(subset=["Resource Group"])
    .groupby("Resource")["Resource Group"]
    .agg(lambda x: x.mode()[0])
    .to_dict()
)

country_map = (
    df.dropna(subset=["Country"])
    .groupby("Resource")["Country"]
    .agg(lambda x: x.mode()[0])
    .to_dict()
)

# Điền giá trị bị thiếu
df["Resource Group"] = df["Resource Group"].fillna(df["Resource"].map(rg_map))
df["Country"] = df["Country"].fillna(df["Resource"].map(country_map))


# ==========================================
# 3. CHUYỂN ĐỔI DURATION THÀNH SỐ GIỜ THẬP PHÂN
# ==========================================
def parse_duration_to_hours(val):
    """Chuyển chuỗi định dạng 'HH:MM' hoặc 'HH:MM:SS' thành số giờ (Float).

    Ví dụ: '7:45' -> 7.75, '8:00' -> 8.0, '0:15' -> 0.25.
    """
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip()
    parts = val_str.split(":")
    if len(parts) == 2:
        hours = int(parts[0])
        minutes = int(parts[1])
        return round(hours + minutes / 60.0, 4)
    elif len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2])
        return round(hours + minutes / 60.0 + seconds / 3600.0, 4)
    try:
        return float(val)
    except (ValueError, TypeError):
        return 0.0


# Tạo cột mới 'Duration_Hours' và giữ lại hoặc ghi đè cột gốc nếu cần
df["Duration_Hours"] = df["Duration"].apply(parse_duration_to_hours)

# ==========================================
# 4. KIỂM TRA LẠI KẾT QUẢ VÀ XUẤT FILE
# ==========================================
print("--- KIỂM TRA GIÁ TRỊ NULL SAU XỬ LÝ ---")
print(
    f"Resource Group nulls: {df['Resource Group'].isnull().sum()}"
)  # Kết quả: 0
print(f"Country nulls:        {df['Country'].isnull().sum()}")  # Kết quả: 0

print("\n--- MẪU DỮ LIỆU DURATION CHUYỂN ĐỔI ---")
print(df[["Duration", "Duration_Hours"]].drop_duplicates().head(10))

# Xuất ra file Excel hoặc CSV sạch để đưa vào Power BI
output_file = r"E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\Sample_Data_Cleaned.xlsx"
df.to_excel(output_file, index=False)
print(f"\nĐã lưu file thành công tại: {output_file}")

--- KIỂM TRA GIÁ TRỊ NULL SAU XỬ LÝ ---
Resource Group nulls: 0
Country nulls:        0

--- MẪU DỮ LIỆU DURATION CHUYỂN ĐỔI ---
   Duration  Duration_Hours
0      8:00            8.00
1      7:45            7.75
2      0:15            0.25
6      0:30            0.50
7      4:00            4.00
8      3:45            3.75
15     7:30            7.50
23     5:00            5.00
24     1:30            1.50
25     6:30            6.50

Đã lưu file thành công tại: E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\Sample_Data_Cleaned.xlsx


In [ ]:
import numpy as np
import pandas as pd

# 1. Đọc dữ liệu từ file Excel (yêu cầu đã cài openpyxl)
file_path = r"E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\Sample Data.xlsx"
df = pd.read_excel(file_path, engine="openpyxl")

# ==========================================
# 2. XỬ LÝ DỮ LIỆU KHUYẾT THIẾU (NULL VALUES)
# ==========================================
rg_map = (
    df.dropna(subset=["Resource Group"])
    .groupby("Resource")["Resource Group"]
    .agg(lambda x: x.mode()[0])
    .to_dict()
)

country_map = (
    df.dropna(subset=["Country"])
    .groupby("Resource")["Country"]
    .agg(lambda x: x.mode()[0])
    .to_dict()
)

df["Resource Group"] = df["Resource Group"].fillna(df["Resource"].map(rg_map))
df["Country"] = df["Country"].fillna(df["Resource"].map(country_map))


# ==========================================
# 3. CHUYỂN ĐỔI DURATION THÀNH SỐ GIỜ THẬP PHÂN
# ==========================================
def parse_duration_to_hours(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip()
    parts = val_str.split(":")
    if len(parts) == 2:
        return round(int(parts[0]) + int(parts[1]) / 60.0, 4)
    elif len(parts) == 3:
        return round(
            int(parts[0]) + int(parts[1]) / 60.0 + int(parts[2]) / 3600.0, 4
        )
    try:
        return float(val)
    except (ValueError, TypeError):
        return 0.0


df["Duration_Hours"] = df["Duration"].apply(parse_duration_to_hours)

# ==========================================
# 4. XUẤT RA TỆP DỮ LIỆU SẠCH (CSV HOẶC EXCEL)
# ==========================================
# Khuyên dùng: Xuất CSV để nạp vào Power BI nhanh và nhẹ nhất
csv_output = (
    r"E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\Sample_Data_Cleaned.csv"
)
df.to_csv(csv_output, index=False, encoding="utf-8-sig")
print(f"Đã lưu thành công tệp CSV: {csv_output}")

# (Tùy chọn) Nếu vẫn cần xuất ra Excel:
# xlsx_output = r"E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\Sample_Data_Cleaned.xlsx"
# df.to_excel(xlsx_output, index=False, engine='openpyxl')

Đã lưu thành công tệp CSV: E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\Sample_Data_Cleaned.csv


In [3]:
import os
from google.cloud import bigquery
import numpy as np
import pandas as pd

# ==========================================
# 1. CẤU HÌNH THÔNG TIN KẾT NỐI BIGQUERY
# ==========================================
csv_file_path = (
    r"E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\Sample_Data_truong.csv"
)
key_path = r"E:\document\DA\BTVN\PBI\PRJ LEGO\xxxxx.json"

project_id = "jda-k1"
dataset_id = "practice_data_pipeline"
table_id = "fact_timesheets_truong"  # Tên bảng sẽ tạo trên BigQuery

full_table_id = f"{project_id}.{dataset_id}.{table_id}"

# Xác thực bằng file JSON Service Account
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = key_path
client = bigquery.Client(project=project_id)

# ==========================================
# 2. ĐỌC VÀ ÉP KIỂU DỮ LIỆU CHẶT CHẼ TRÊN PANDAS
# ==========================================
df = pd.read_csv(csv_file_path)

# Phòng trường hợp file CSV chưa có cột Duration_Hours thì tự động tính bổ sung
if "Duration_Hours" not in df.columns and "Duration" in df.columns:

    def parse_duration(val):
        if pd.isna(val):
            return 0.0
        parts = str(val).strip().split(":")
        if len(parts) == 2:
            return round(int(parts[0]) + int(parts[1]) / 60.0, 4)
        elif len(parts) == 3:
            return round(
                int(parts[0]) + int(parts[1]) / 60.0 + int(parts[2]) / 3600.0, 4
            )
        return float(val)

    df["Duration_Hours"] = df["Duration"].apply(parse_duration)

# Chuẩn hóa tên cột: thay khoảng trắng bằng dấu gạch dưới (_) để tránh lỗi cú pháp SQL & Power BI
df.columns = [col.strip().replace(" ", "_") for col in df.columns]

# A. Ép kiểu DATE (Chỉ lấy phần ngày YYYY-MM-DD, tránh dính 00:00:00)
date_cols = ["Date_Created", "Timesheet_Date", "Week_Of"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce").dt.date

# B. Ép kiểu DATETIME (Giữ nguyên ngày + giờ phút giây, không gắn timezone)
datetime_cols = ["Date_Time_Last_Amended", "Date_Time_Approved"]
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# C. Ép kiểu FLOAT64 (Số thực cho giờ công)
df["Duration_Hours"] = pd.to_numeric(
    df["Duration_Hours"], errors="coerce"
).astype("float64")

# D. Ép kiểu STRING (Dùng 'string' của Pandas để giữ nguyên giá trị NULL thay vì biến thành chữ 'nan')
string_cols = [
    "Resource_Group",
    "Projects",
    "Resource",
    "Country",
    "Time_Approver",
    "Approved_By",
    "Duration",
    "Approval_Status",
    "Hour_Type_1",
    "Hour_Type_2",
    "Type",
]
for col in string_cols:
    df[col] = df[col].astype("string")

# ==========================================
# 3. ĐỊNH NGHĨA SCHEMA CỨNG CHO BIGQUERY
# ==========================================
bq_schema = [
    bigquery.SchemaField("Date_Created", "DATE"),
    bigquery.SchemaField("Timesheet_Date", "DATE"),
    bigquery.SchemaField("Resource_Group", "STRING"),
    bigquery.SchemaField("Projects", "STRING"),
    bigquery.SchemaField("Resource", "STRING"),
    bigquery.SchemaField("Country", "STRING"),
    bigquery.SchemaField("Time_Approver", "STRING"),
    bigquery.SchemaField("Approved_By", "STRING"),
    bigquery.SchemaField("Duration", "STRING"),
    bigquery.SchemaField("Approval_Status", "STRING"),
    bigquery.SchemaField("Hour_Type_1", "STRING"),
    bigquery.SchemaField("Hour_Type_2", "STRING"),
    bigquery.SchemaField("Type", "STRING"),
    bigquery.SchemaField("Date_Time_Last_Amended", "DATETIME"),
    bigquery.SchemaField("Date_Time_Approved", "DATETIME"),
    bigquery.SchemaField("Week_Of", "DATE"),
    bigquery.SchemaField("Duration_Hours", "FLOAT64"),
]

# Đảm bảo thứ tự cột của DataFrame khớp hoàn toàn với thứ tự trong bq_schema
ordered_cols = [field.name for field in bq_schema]
df = df[ordered_cols]

# ==========================================
# 4. ĐẨY DỮ LIỆU LÊN BIGQUERY
# ==========================================
# Tạo dataset nếu chưa tồn tại
dataset_ref = bigquery.Dataset(f"{project_id}.{dataset_id}")
dataset_ref.location = "US"  # Có thể đổi thành "asia-southeast1" nếu muốn đặt server ở Singapore
client.create_dataset(dataset_ref, exists_ok=True)

job_config = bigquery.LoadJobConfig(
    schema=bq_schema,
    write_disposition="WRITE_TRUNCATE",  # Ghi đè sạch bảng nếu chạy lại code nhiều lần
)

print(f"Đang đẩy {len(df):,} dòng dữ liệu lên {full_table_id}...")
job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
job.result()  # Đợi job hoàn tất

# ==========================================
# 5. KIỂM TRA SCHEMA THỰC TẾ TRÊN BIGQUERY
# ==========================================
table = client.get_table(full_table_id)
print(
    f"\nThành công! Bảng '{table.table_id}' hiện có {table.num_rows:,} dòng."
)
print("--- SCHEMA TRÊN BIGQUERY ---")
for field in table.schema:
    print(f"  - {field.name:<25} : {field.field_type}")

Đang đẩy 103,961 dòng dữ liệu lên jda-k1.practice_data_pipeline.fact_timesheets_truong...


c:\Users\Admin\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(



Thành công! Bảng 'fact_timesheets_truong' hiện có 103,961 dòng.
--- SCHEMA TRÊN BIGQUERY ---
  - Date_Created              : DATE
  - Timesheet_Date            : DATE
  - Resource_Group            : STRING
  - Projects                  : STRING
  - Resource                  : STRING
  - Country                   : STRING
  - Time_Approver             : STRING
  - Approved_By               : STRING
  - Duration                  : STRING
  - Approval_Status           : STRING
  - Hour_Type_1               : STRING
  - Hour_Type_2               : STRING
  - Type                      : STRING
  - Date_Time_Last_Amended    : DATETIME
  - Date_Time_Approved        : DATETIME
  - Week_Of                   : DATE
  - Duration_Hours            : FLOAT


In [4]:
import os
from google.cloud import bigquery
import pandas as pd

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN VÀ KẾT NỐI BIGQUERY
# ==========================================
rate_csv_path = r"E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\rate_truong.csv"
contact_csv_path = (
    r"E:\document\DA\BTVN\PBI\PRJ LEGO\dataset\contact_truong.csv"
)
key_path = r"E:\document\DA\BTVN\PBI\PRJ LEGO\xxxx.json"

project_id = "jda-k1"
dataset_id = "practice_data_pipeline"

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = key_path
client = bigquery.Client(project=project_id)

# ==========================================
# 2. XỬ LÝ VÀ ĐẨY BẢNG RATE (dim_resource_rate)
# ==========================================
df_rate = pd.read_csv(rate_csv_path, sep=";", decimal=",")
df_rate.columns = [
    c.strip().replace(" ", "_").replace("/", "_") for c in df_rate.columns
]

# Ép kiểu chặt chẽ trên Pandas
df_rate["Resource_Group"] = df_rate["Resource_Group"].astype("string")
df_rate["Rate_Per_Hr"] = pd.to_numeric(
    df_rate["Rate_Per_Hr"], errors="coerce"
).astype("float64")

rate_schema = [
    bigquery.SchemaField("Resource_Group", "STRING"),
    bigquery.SchemaField("Rate_Per_Hr", "FLOAT64"),
]

rate_table_id = f"{project_id}.{dataset_id}.dim_resource_rate"
job_rate = client.load_table_from_dataframe(
    df_rate[["Resource_Group", "Rate_Per_Hr"]],
    rate_table_id,
    job_config=bigquery.LoadJobConfig(
        schema=rate_schema, write_disposition="WRITE_TRUNCATE"
    ),
)
job_rate.result()

# ==========================================
# 3. XỬ LÝ VÀ ĐẨY BẢNG CONTRACT (dim_country_contract)
# ==========================================
df_contact = pd.read_csv(contact_csv_path, sep=";", decimal=",")
df_contact.columns = [
    c.strip().replace(" ", "_").replace("/", "_") for c in df_contact.columns
]

# Ép kiểu chặt chẽ trên Pandas
df_contact["Country"] = df_contact["Country"].astype("string")
df_contact["Contractual_Hrs_Week"] = pd.to_numeric(
    df_contact["Contractual_Hrs_Week"], errors="coerce"
).astype("float64")

contact_schema = [
    bigquery.SchemaField("Country", "STRING"),
    bigquery.SchemaField("Contractual_Hrs_Week", "FLOAT64"),
]

contact_table_id = f"{project_id}.{dataset_id}.dim_country_contract"
job_contact = client.load_table_from_dataframe(
    df_contact[["Country", "Contractual_Hrs_Week"]],
    contact_table_id,
    job_config=bigquery.LoadJobConfig(
        schema=contact_schema, write_disposition="WRITE_TRUNCATE"
    ),
)
job_contact.result()

# ==========================================
# 4. KIỂM TRA SCHEMA VÀ DỮ LIỆU SAU KHI NẠP
# ==========================================
for tbl_id in [rate_table_id, contact_table_id]:
    table = client.get_table(tbl_id)
    print(f"\nThành công: {table.table_id} ({table.num_rows} dòng)")
    for field in table.schema:
        print(f"  - {field.name:<25} : {field.field_type}")

c:\Users\Admin\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(
c:\Users\Admin\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(



Thành công: dim_resource_rate (6 dòng)
  - Resource_Group            : STRING
  - Rate_Per_Hr               : FLOAT

Thành công: dim_country_contract (3 dòng)
  - Country                   : STRING
  - Contractual_Hrs_Week      : FLOAT
